# 🧠 DES Hallucination Proxy: 3D Embedding Visualization Lab

This notebook provides an interactive 3D laboratory to visualize model disagreement and hallucinations in semantic space.

### 🔬 Objectives:
1. **Semantic Clustering**: Visualize how different model architectures (Llama, GPT, Qwen, etc.) cluster for the same question.
2. **Hallucination Detection**: Identify "scattered" response patterns that correlate with high DES scores.
3. **Ensemble Analysis**: See where the "Wisdom of the Crowd" succeeds and where it fails.

In [ ]:
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go
import umap
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded.")

## 1. Load Scored Results
We load the results generated by the DES pipeline (`data/results/scored_results.jsonl`).

In [ ]:
DATA_PATH = "../data/results/scored_results.jsonl"

def load_results(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

df = load_results(DATA_PATH)
print(f"Loaded {len(df)} questions.")
df.head(2)

## 2. Interactive 3D Visualization Logic
The core function to visualize a single question.

In [ ]:
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

def visualize_question_3d(qid_index):
    row = df.iloc[qid_index]
    question = row['question']
    responses_dict = row.get('model_responses', {})
    des_score = row.get('DES', 0.0)
    
    ground_truth_list = row.get('correct_answers', ['N/A'])
    correct_answer = str(ground_truth_list[0]) if isinstance(ground_truth_list, list) else str(ground_truth_list)

    model_names = list(responses_dict.keys())
    texts = [str(resp_obj.get('answer', '')) for resp_obj in responses_dict.values()]
    
    texts.append(correct_answer)
    model_names.append("GROUND_TRUTH")

    embeddings = model_embed.encode(texts)
    reducer = umap.UMAP(n_components=3, n_neighbors=2, min_dist=0.1, random_state=42)
    embed_3d = reducer.fit_transform(embeddings)

    plot_df = pd.DataFrame(embed_3d, columns=['x', 'y', 'z'])
    plot_df['Model'] = model_names
    plot_df['Response'] = texts
    plot_df['Type'] = ['Model' if m != 'GROUND_TRUTH' else 'Truth' for m in model_names]
    
    fig = px.scatter_3d(
        plot_df, x='x', y='y', z='z', 
        color='Model', 
        symbol='Type',
        hover_data=['Response'],
        title=f"QID {qid_index}: {question[:80]}...<br>DES Score: {des_score:.4f}",
        labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'z': 'UMAP 3'}
    )
    
    fig.update_traces(marker=dict(size=8), selector=dict(mode='markers'))
    fig.show()

print("🚀 Visualization function ready.")

## 3. Geometric Scanners (Advanced Analysis)

### A. Consensus Lie Detector
Finds cases where models cluster tightly (Agreement) but are far from the truth (Hallucination). This identifies the Failure Mode of ensembling.

In [ ]:
def find_consensus_lies(num_samples=10):
    """
    Heuristic: Find cases with LOW Semantic DES (Agreement) 
    but LOW Correctness Flags (All are wrong).
    """
    # filter where no model is correct
    wrong_indices = df[df['correctness_flags'].apply(lambda x: not any(x.values()))].index
    
    # sort by lowest semantic DES (tightest cluster of models)
    consensus_indices = df.loc[wrong_indices].sort_values('semantic_DES').head(num_samples).index
    
    print(f"Detected {len(consensus_indices)} cases where models likely agreed on a wrong answer.")
    return consensus_indices

lies = find_consensus_lies()
if len(lies) > 0:
    print(f"Try visualizing QID: {lies[0]}")

### B. Model Family Clustering
Visualize multiple questions to see if Llama, Qwen, etc. stay in their own "territories."

In [ ]:
def visualize_families_3d(qid_range=range(0, 100, 10)):
    """
    Plots multiple questions at once to show architectural persistent behavior.
    """
    all_embeds = []
    all_names = []
    all_qids = []
    
    for qid in tqdm(qid_range):
        row = df.iloc[qid]
        responses_dict = row.get('model_responses', {})
        texts = [str(resp_obj.get('answer', '')) for resp_obj in responses_dict.values()]
        
        embeds = model_embed.encode(texts)
        all_embeds.extend(embeds)
        all_names.extend(list(responses_dict.keys()))
        all_qids.extend([f"Q{qid}"] * len(texts))
    
    # Reduce everything together
    reducer = umap.UMAP(n_components=3, random_state=42)
    embed_3d = reducer.fit_transform(all_embeds)
    
    plot_df = pd.DataFrame(embed_3d, columns=['x', 'y', 'z'])
    plot_df['Model'] = all_names
    plot_df['Question'] = all_qids
    
    # Map into broad families
    plot_df['Family'] = plot_df['Model'].apply(lambda x: 'Meta' if 'llama' in x else ('Alibaba' if 'qwen' in x else ('Mistral' if 'mistral' in x else 'Other')))

    fig = px.scatter_3d(
        plot_df, x='x', y='y', z='z', 
        color='Family',
        hover_data=['Model'],
        title=f"Geometric Diversity: Architectural Clustering for {len(qid_range)} Questions"
    )
    fig.show()

visualize_families_3d()

## 4. Launch specific QID
Enter an index (0 to 2016) to visualize the semantic disagreement for that specific question.

In [ ]:
visualize_question_3d(142)